# Simplicio 27B · Serving (vLLM + completion gate + Cloudflare Tunnel)
Runtime > Change runtime type > **G4 (NVIDIA RTX PRO 6000, 96 GB)**. Os pesos BF16 ocupam 55,6 GB. Depois: Run all.

`deploy/serve_colab.py` roda `deploy/serve_vllm.sh` (o único comando de serve, `MAX_MODEL_LEN` de `deploy/context.env`), coloca o gate de `finish_reason`+`usage` na frente, abre o túnel, faz um POST de prova com os dois ids e só então chama `set_upstream.php`. No exit o upstream é limpo.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip install -q vllm==0.31.0
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64 && echo "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db  cloudflared" | sha256sum -c - && chmod +x cloudflared || { rm -f cloudflared; exit 1; }


In [ ]:
import os, pathlib, urllib.request
SIMPLICIO_SHA = "58e1c17d36db3a6014a72db30c3421a61194ec0e"  # commit de main com deploy/; troque quando deploy/ mudar
RAW = os.environ.get("SIMPLICIO_RAW", f"https://raw.githubusercontent.com/simpletibr/simplicio-27b/{SIMPLICIO_SHA}")
pathlib.Path("deploy").mkdir(exist_ok=True)
for name in ("context.env", "completion_gate.py", "serve_colab.py", "serve_vllm.sh"):
    urllib.request.urlretrieve(f"{RAW}/deploy/{name}", f"deploy/{name}")
print("deploy files ready")


In [ ]:
import os
from getpass import getpass
from google.colab import userdata

try:
    key = userdata.get("SIMPLETI_ADMIN_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
    key = getpass("Chave admin do gateway: ")
key = key.strip()
if not key:
    raise SystemExit("SIMPLETI_ADMIN_KEY vazia")
os.environ["SIMPLETI_ADMIN_KEY"] = key
print("chave admin carregada")


In [ ]:
!CLOUDFLARED_BIN=./cloudflared python3 deploy/serve_colab.py
